# 02. 전이학습 — 사전학습 모델 활용하기

앞 노트북에서는 CNN을 처음부터(from scratch) 학습했습니다.
이번에는 **ImageNet으로 미리 학습된 ResNet18** 을 가져와 CIFAR-10에 맞게 **미세조정(fine-tuning)** 합니다.

이것이 **전이학습(Transfer Learning)** 입니다. 대규모 데이터로 학습된 모델의 지식을 가져오면,
적은 데이터와 짧은 학습으로도 처음부터 학습한 모델보다 **높은 정확도**를 얻을 수 있습니다.

실습 흐름:
1. **데이터 준비** — ResNet 입력 형식에 맞게 전처리
2. **사전학습 모델 불러오기** — ResNet18 + 분류기 교체
3. **미세조정 학습**
4. **평가 및 scratch CNN과 비교**

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights
import matplotlib.pyplot as plt

plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('사용 장치:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 1. 데이터 준비

ResNet18은 ImageNet(224×224, 특정 평균·표준편차로 정규화)으로 학습되었습니다.
사전학습 가중치를 제대로 활용하려면 **입력을 그 형식에 맞춰야** 합니다.

- CIFAR-10의 32×32 이미지를 **96×96으로 확대**(교육용 절충 크기 — 224는 학습이 느림)
- ImageNet 평균·표준편차로 정규화

In [ ]:
img_size = 96
transform = transforms.Compose([
    transforms.Resize(img_size),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),  # ImageNet 통계
])

train_set = torchvision.datasets.CIFAR10(root='/workspace/data', train=True,
                                         download=True, transform=transform)
test_set = torchvision.datasets.CIFAR10(root='/workspace/data', train=False,
                                        download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2)
print('데이터 준비 완료')

## 2. 사전학습 모델 불러오기

`ResNet18_Weights.DEFAULT` 로 ImageNet 사전학습 가중치를 함께 받습니다 (처음 실행 시 자동 다운로드).

ResNet18의 마지막 분류기(`fc`)는 ImageNet 1000개 클래스용이므로,
**CIFAR-10의 10개 클래스에 맞게 교체**합니다. 나머지 합성곱 층(특징 추출부)은 사전학습된 가중치를 그대로 물려받습니다.

In [ ]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 10)   # 1000 -> 10 클래스
model = model.to(device)

print('마지막 분류기 교체 완료:', model.fc)

## 3. 미세조정 학습

전체 네트워크를 학습하되, 사전학습된 특징 위에서 시작하므로 **빠르게 수렴**합니다.
교육용으로 5 에폭만 학습합니다 (scratch CNN의 10 에폭보다 적습니다).

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f'[{epoch+1}/{EPOCHS}] 평균 손실: {running_loss/len(train_loader):.4f}')

print('학습 완료')

## 4. 평가 및 비교

In [ ]:
model.eval()
correct = total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

acc = 100 * correct / total
print(f'전이학습 테스트 정확도: {acc:.2f}%')

### 정리

- ImageNet 사전학습 ResNet18을 가져와 분류기만 교체하고 **5 에폭**만 미세조정했습니다.
- 직접 만든 CNN(10 에폭, 70% 안팎)보다 **더 적은 학습으로 더 높은 정확도**(보통 85% 이상)를 얻습니다.
- 이것이 전이학습의 핵심입니다 — **남이 큰 데이터로 학습해 둔 특징을 재활용**하면, 내 데이터가 적어도 좋은 성능을 빠르게 얻을 수 있습니다.
- 이 컨테이너(`edu-cv-pytorch`)의 정체성인 **CNN 전이학습** 실습입니다.